# Voice Conversion Humanization Notebook

This notebook processes raw TTS audio files using voice conversion techniques to add natural characteristics.

## Method: Voice Conversion (RVC/SoVITS)
- Uses RVC (Retrieval-based Voice Conversion) for fast processing
- Applies light voice conversion to add natural characteristics
- Preserves speaker identity while enhancing naturalness
- Fast inference suitable for batch processing

## Input/Output:
- **Input**: `outputs/raw/{batch_run_id}/{speaker_id}/{prompt_id}.wav`
- **Output**: `outputs/voice_conversion/{batch_run_id}/{speaker_id}/{prompt_id}.wav`



In [ ]:
# Cell 1: Environment Setup and Path Detection

import os
from pathlib import Path

# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("VOICE CONVERSION HUMANIZATION - Environment Setup")
print("=" * 80)

if IN_COLAB:
    print("Detected: Google Colab environment")
    print("\nMounting Google Drive...")
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Libri_Vibevoice")
else:
    print("Detected: Local environment")
    BASE_DIR = Path.cwd()
    if "Libri_Vibevoice" not in str(BASE_DIR):
        # Fallback: use the known path structure
        BASE_DIR = Path(r"G:\My Drive\Libri_Vibevoice")
    print(f"Using BASE_DIR: {BASE_DIR}")

# Set up paths
INPUT_DIR = BASE_DIR / "outputs" / "raw"  # Read from raw outputs
OUTPUT_DIR = BASE_DIR / "outputs" / "voice_conversion"  # Save to voice_conversion directory

print(f"\nBASE_DIR: {BASE_DIR}")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

# Verify input directory exists
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory not found at: {INPUT_DIR}")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n✓ Setup complete. Input directory found: {INPUT_DIR}")
print(f"✓ Humanized outputs will be saved to: {OUTPUT_DIR}")



In [ ]:
# Cell 2: Install Dependencies

import subprocess
import sys

print("=" * 80)
print("Installing Required Packages")
print("=" * 80)

packages = [
    "torch>=2.0.0",
    "librosa>=0.10.0",
    "soundfile>=0.12.0",
    "numpy>=1.24.0",
    "tqdm>=4.65.0",
]

# Note: RVC/SoVITS installation
# For Colab, we'll clone RVC repository
# For local, user should install RVC manually
if IN_COLAB:
    print("Cloning RVC repository...")
    try:
        subprocess.run(["git", "clone", "https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git", "/content/RVC"], check=False)
        sys.path.insert(0, "/content/RVC")
        print("✓ RVC repository cloned")
    except Exception as e:
        print(f"⚠ Warning: Could not clone RVC repository: {e}")
        print("Will use simplified voice conversion approach")
else:
    print("Note: For local installation, install RVC manually:")
    print("  git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git")
    print("  cd Retrieval-based-Voice-Conversion-WebUI")
    print("  pip install -r requirements.txt")

for package in packages:
    try:
        print(f"Installing {package}...")
        if IN_COLAB:
            subprocess.run([sys.executable, "-m", "pip", "install", package, "--quiet"], check=True)
        else:
            subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"⚠ Warning: Failed to install {package}: {e}")

print("\n✓ Dependencies installation complete")



In [ ]:
# Cell 3: Load Voice Conversion Model

import torch

print("=" * 80)
print("Loading Voice Conversion Model")
print("=" * 80)

# Device selection
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Using Apple Silicon GPU")
else:
    DEVICE = "cpu"
    print("Using CPU")

# Note: RVC model loading requires specific setup
# This is a simplified version - adjust based on actual RVC API
print("\nNote: Voice conversion model loading requires:")
print("  1. Pre-trained RVC model files")
print("  2. Initializing the RVC pipeline")
print("  3. Loading feature extractors")

# Placeholder for RVC initialization
# In practice, you would:
# from rvc import RVC
# model = RVC(model_path="path/to/model")
# model.to(DEVICE)

print("⚠ Voice conversion model loading needs to be implemented based on actual RVC API")
print("For now, using simplified processing approach")
print("Please refer to RVC documentation for proper initialization")



In [ ]:
# Cell 4: Voice Conversion Processing Functions

import librosa
import soundfile as sf
import numpy as np

def extract_voice_features(audio, sr=24000):
    """
    Extract voice features for conversion.
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
    
    Returns:
        Voice features dictionary
    """
    # Extract basic features
    # In practice, RVC uses specific feature extractors
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    
    return {
        'mfcc': mfcc,
        'spectral_centroid': spectral_centroid,
    }


def apply_voice_conversion(audio, sr=24000):
    """
    Apply voice conversion to add natural characteristics.
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
    
    Returns:
        Converted audio with natural characteristics
    """
    # Extract features
    features = extract_voice_features(audio, sr)
    
    # Apply light voice conversion
    # Note: This is a placeholder - actual RVC implementation would:
    # 1. Extract speaker embeddings
    # 2. Apply conversion model
    # 3. Re-synthesize with natural characteristics
    
    # For now, apply light processing to enhance naturalness
    # (This is a simplified version - replace with actual RVC calls)
    
    # Light spectral enhancement
    stft = librosa.stft(audio)
    magnitude = np.abs(stft)
    phase = np.angle(stft)
    
    # Apply gentle spectral smoothing
    from scipy import ndimage
    magnitude_smooth = ndimage.gaussian_filter(magnitude, sigma=0.5)
    
    # Reconstruct audio
    stft_enhanced = magnitude_smooth * np.exp(1j * phase)
    audio_enhanced = librosa.istft(stft_enhanced)
    
    return audio_enhanced, sr


def process_audio_file(input_path, output_path):
    """
    Process a single audio file through voice conversion.
    
    Args:
        input_path: Path to input audio file
        output_path: Path to save processed audio
    
    Returns:
        True if successful, False otherwise
    """
    try:
        # Load audio
        audio, sr = librosa.load(str(input_path), sr=None)
        
        # Apply voice conversion
        processed_audio, processed_sr = apply_voice_conversion(audio, sr)
        
        # Ensure output directory exists
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Save processed audio
        sf.write(str(output_path), processed_audio, processed_sr)
        
        return True
    except Exception as e:
        print(f"  ✗ Error processing {input_path}: {e}")
        import traceback
        traceback.print_exc()
        return False

print("✓ Voice conversion processing functions defined")
print("⚠ Note: Actual RVC API calls need to be implemented for full functionality")



In [ ]:
# Cell 5: Batch Processing

from pathlib import Path
from tqdm import tqdm

print("=" * 80)
print("Batch Processing - Humanizing Audio Files with Voice Conversion")
print("=" * 80)

# Configuration: Process all batch runs or specify specific ones
BATCH_RUNS_TO_PROCESS = 'all'  # Change to list like ['25094444', '3ac81b81'] for specific runs

# Collect all WAV files to process
files_to_process = []

if BATCH_RUNS_TO_PROCESS == 'all':
    # Get all batch run directories
    if INPUT_DIR.exists():
        batch_runs = [d for d in INPUT_DIR.iterdir() if d.is_dir()]
    else:
        batch_runs = []
else:
    # Process only specified batch runs
    batch_runs = [INPUT_DIR / run_id for run_id in BATCH_RUNS_TO_PROCESS if (INPUT_DIR / run_id).exists()]

print(f"Found {len(batch_runs)} batch run(s) to process")

for batch_run_dir in batch_runs:
    batch_run_id = batch_run_dir.name
    
    # Find all speaker directories
    for speaker_dir in batch_run_dir.iterdir():
        if not speaker_dir.is_dir():
            continue
        
        speaker_id = speaker_dir.name
        
        # Find all WAV files
        for wav_file in speaker_dir.glob("*.wav"):
            # Create corresponding output path maintaining structure
            relative_path = wav_file.relative_to(INPUT_DIR)
            output_path = OUTPUT_DIR / relative_path
            
            files_to_process.append((wav_file, output_path))

print(f"\nFound {len(files_to_process)} WAV files to process")

if len(files_to_process) == 0:
    print("⚠ No files found to process. Check your INPUT_DIR and BATCH_RUNS_TO_PROCESS configuration.")
else:
    # Process files with progress bar
    successful = 0
    failed = 0
    
    for input_path, output_path in tqdm(files_to_process, desc="Processing with Voice Conversion"):
        if process_audio_file(input_path, output_path):
            successful += 1
        else:
            failed += 1
    
    print("\n" + "=" * 80)
    print("Processing Complete")
    print("=" * 80)
    print(f"✓ Successfully processed: {successful} files")
    if failed > 0:
        print(f"✗ Failed: {failed} files")
    print(f"\nHumanized files saved to: {OUTPUT_DIR}")



In [ ]:
# Cell 6: Test Single File (Optional)

print("=" * 80)
print("Test Single File")
print("=" * 80)

# Select a test file (modify these as needed)
TEST_BATCH_RUN_ID = "25094444"  # Change to test different batch run
TEST_SPEAKER_ID = "196"  # Change to test different speaker
TEST_FILENAME = "multi_turn_base_19.wav"  # Change to test different file

test_input_path = INPUT_DIR / TEST_BATCH_RUN_ID / TEST_SPEAKER_ID / TEST_FILENAME
test_output_path = OUTPUT_DIR / TEST_BATCH_RUN_ID / TEST_SPEAKER_ID / TEST_FILENAME.replace('.wav', '_test.wav')

if not test_input_path.exists():
    print(f"⚠ Test file not found: {test_input_path}")
    print("Please update TEST_BATCH_RUN_ID, TEST_SPEAKER_ID, or TEST_FILENAME above.")
else:
    print(f"Testing with: {test_input_path}")
    
    # Process the test file
    if process_audio_file(test_input_path, test_output_path):
        print(f"✓ Test file processed successfully")
        print(f"  Original: {test_input_path}")
        print(f"  Humanized: {test_output_path}")
        
        # Play audio if in Colab
        if IN_COLAB:
            from IPython.display import Audio, display
            print("\nPlaying original audio:")
            display(Audio(str(test_input_path)))
            print("\nPlaying humanized audio:")
            display(Audio(str(test_output_path)))
        else:
            print("\nTo listen to the audio, open the files:")
            print(f"  Original: {test_input_path}")
            print(f"  Humanized: {test_output_path}")
    else:
        print("✗ Failed to process test file")

